# PNG Connectivity Test

Validates **Databricks serverless → private destination** connectivity through a **Private Network Gateway (PNG)**.

This notebook attempts a TCP connection from serverless compute to a **PostgreSQL** instance (`example.com`, port **5432**) that is reachable only over the PNG, addressed by its private hostname.

## Prerequisites
- Run on **serverless** compute in a workspace whose **NCC has the PNG attached**.
- The target host is listed in the PNG **`destinations`**.
- A **private DNS resolver** (reached via the PNG) resolves the private hostname.

## Why force IPv6?
Databricks serverless is IPv6-internal and uses **DNS64/NAT64**: private IPv4 destinations are resolved as synthesized IPv6 addresses (prefix `64:ff9b::/…`) and translated back to IPv4 on egress through the gateway. The socket test therefore forces `AF_INET6` — an `AF_INET` (IPv4) lookup would not return an address.

## Target endpoint
The private JDBC endpoint under test (resolved and routed through the PNG):

In [0]:
jdbc://example.com:5432/

## DNS resolution check
Confirm the hostname resolves through the PNG's private DNS resolver. Because serverless uses **DNS64/NAT64**, the name resolves to a **synthesized IPv6 address** (prefix `64:ff9b::/…`) that NAT64 maps to the real IPv4 target on egress — note there is **no A (IPv4) record**, only the synthesized AAAA.

In [ ]:
%sh
nslookup example.com

;; Got recursion not available from 192.168.200.6
Server:		192.168.200.6
Address:	192.168.200.6#53

Non-authoritative answer:
;; Got recursion not available from 192.168.200.6
Name:	example.com
Address: 64:ff9b:1::100:a04:d04


## TCP reachability test
Resolves the private hostname over IPv6 and attempts a TCP connect to port 5432.

- **`SUCCESS`** → the full PNG path works end-to-end (DNS resolution → routing → target firewall/NSG).
- **`FAIL` (timeout)** → a break somewhere on the path: missing PNG destination, routing/peering, the private DNS resolver, or the target's security group / NSG not allowing 5432.

In [0]:
import socket

host = "example.com"
port = 5432

# force IPv6 — PNG targets resolve to an IPv6 address
for family in (socket.AF_INET6,):
    addrs = socket.getaddrinfo(host, port, family, socket.SOCK_STREAM) 
    for _, socktype, proto, _, sockaddr in addrs: 
        s = socket.socket(family, socktype, proto); 
        s.settimeout(5) 
        try: 
            s.connect(sockaddr);
            print("SUCCESS", sockaddr) 
        except Exception as e: 
            print("FAIL", sockaddr, e) 
        finally: 
            s.close()

SUCCESS ('64:ff9b:1::100:a04:d04', 5432, 0, 0)
